# 📊 Exploratory Data Analysis (EDA) - Crypto Pulse

This notebook focuses on analyzing the crypto price data and news/social data to find patterns and correlations.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import json
import glob
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = [12, 6]

## 2. Load Price Data

We have historical kline data in `data/historical/`.

In [ ]:
def load_kline_data(symbol):
    file_path = f'../data/historical/{symbol.lower()}usdt_raw_klines.json'
    if not os.path.exists(file_path):
        # Try without the 'usdt' part if not found
        file_path = f'../data/historical/{symbol.lower()}_raw_klines.json'
        if not os.path.exists(file_path):
            return None
    
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume', 
        'close_time', 'quote_asset_volume', 'number_of_trades', 
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
    ])
    
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
    
    return df

btc_df = load_kline_data('btc')
if btc_df is not None:
    print("BTC Data Loaded Successfully.")
    display(btc_df.head())

## 3. Price & Volume Analysis

Detailed trend analysis for Bitcoin.

In [ ]:
if btc_df is not None:
    fig, ax1 = plt.subplots(figsize=(14, 7))

    ax1.set_xlabel('Date')
    ax1.set_ylabel('Price (USDT)', color='tab:orange')
    ax1.plot(btc_df['open_time'], btc_df['close'], color='tab:orange', label='Price')
    ax1.tick_params(axis='y', labelcolor='tab:orange')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Volume', color='tab:blue')
    ax2.bar(btc_df['open_time'], btc_df['volume'], color='tab:blue', alpha=0.3, label='Volume')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    plt.title('Bitcoin Price and Volume Trend')
    fig.tight_layout()
    plt.show()

## 4. Multi-Currency Correlation

Understanding how different cryptocurrencies move together.

In [ ]:
symbols = ['btc', 'eth', 'bnb', 'sol', 'ada', 'xrp', 'dot', 'doge', 'matic', 'link']
all_prices = {}

for s in symbols:
    df = load_kline_data(s)
    if df is not None:
        # Align by date and use closing price
        all_prices[s.upper()] = df.set_index('open_time')['close']

if all_prices:
    corr_df = pd.DataFrame(all_prices).corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_df, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Crypto Price Correlation Matrix')
    plt.show()
else:
    print("No data available for correlation matrix.")

## 5. Load News/Social Data (Local Backups)

In [ ]:
def load_raw_news():
    files = glob.glob('../data/raw/news/*.json')
    all_articles = []
    for f in files:
        with open(f, 'r', encoding='utf-8') as file:
            all_articles.extend(json.load(file))
    
    if not all_articles:
        return pd.DataFrame()
        
    df = pd.DataFrame(all_articles)
    if 'published_at' in df.columns:
        df['published_at'] = pd.to_datetime(df['published_at'])
    return df

news_df = load_raw_news()
if not news_df.empty:
    print(f"Loaded {len(news_df)} articles.")
    display(news_df.head())
else:
    print("No news data found yet.")

## 6. Sentiment & Word Analysis (Placeholder for Milestone 2)

In the next milestone, we will apply FinBERT to these titles.

In [ ]:
if not news_df.empty:
    # Simple Word Frequency analysis
    from collections import Counter
    import re
    
    all_titles = " ".join(news_df['title'].astype(str))
    words = re.findall(r'\w+', all_titles.lower())
    # Remove common stopwords (simple version)
    stopwords = {'the', 'a', 'to', 'in', 'of', 'and', 'is', 'for', 'on', 'with', 'at', 'by', 'an', 'be', 'this', 'that', 'from', 'as'}
    filtered_words = [w for w in words if w not in stopwords and len(w) > 3]
    
    word_counts = Counter(filtered_words).most_common(20)
    word_df = pd.DataFrame(word_counts, columns=['word', 'count'])
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='count', y='word', data=word_df, palette='viridis')
    plt.title('Top 20 Words in News Titles')
    plt.show()